# Authors
Kseniia Ostudina (kseniia.ostudina@universite-paris-saclay.fr)

Marina Hornero Merino (marina.hornero-merino@universite-paris-saclay.fr)

Julie Dornat (julie.dornat@universite-paris-saclay.fr)

In [1]:
!pip install gensim
!pip install fasttext

  Using cached fasttext-0.9.3.tar.gz (73 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build fasttext


  error: subprocess-exited-with-error
  
  × Building wheel for fasttext (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [134 lines of output]
      C:\Users\jdorn\AppData\Local\Temp\pip-build-env-rc9xxr4e\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: fasttext).
      
              Available configuration options are listed in:
              https://setuptools.pypa.io/en/latest/userguide/declarative_config.html
      
              This deprecation is overdue, please update your project and remove depr

# Imports

In [2]:
import urllib.request
import zipfile
import fasttext
import os
import time

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.word2vec import LineSentence
from scipy.spatial import distance
from collections import Counter

import torch
import sys
from torch import nn
from torch.optim import Adam

from tqdm import tqdm
from torch.utils.data  import TensorDataset, DataLoader
from sklearn.metrics import classification_report
import torch.nn.functional as F
import argparse
import subprocess

ImportError: cannot import name 'Embedding' from 'torch.nn.modules.sparse' (c:\Users\jdorn\Documents\M2\NLP Today\.venv\Lib\site-packages\torch\nn\modules\sparse.py)

In [ ]:
# Extract data

# Define the URL and the target filename
url = 'https://perso.limsi.fr/neveol/TP_ISD2020.zip'
zip_filename = 'TP_ISD2020.zip'

# Download the file from the link
urllib.request.urlretrieve(url, zip_filename)

# Extract the contents into a folder named 'extracted_data'
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('extracted_data')

print('Download and extraction complete!')

Download and extraction complete!


In [ ]:
# Create output directory
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_DIR = "extracted_data"
PRESS_DIR = f"{DATA_DIR}/QUAERO_FrenchPress"
MED_DIR = f"{DATA_DIR}/QUAERO_FrenchMed"
WORDS = ["patient", "traitement", "maladie", "solution", "jaune"]


# FastText Embedding (CBow)


In [ ]:
class FastText_Embedding():
  def __init__(self,
               emb_model="cbow",
               model_path=f"{OUTPUT_DIR}/fattest_embeddings.bin",
               load_model=False):
    self.emb_model = emb_model
    self.model_path = model_path
    if load_model and os.path.exists(self.model_path):
      self.model = fasttext.load_model(model_path)

  def fit(self, train_set: str = "QUAERO_FrenchMed_traindev.ospl"):
    self.model = fasttext.train_unsupervised(
        train_set,
        model=self.emb_model,
        dim=100,
        minCount=1,
    )
    self.model.save_model(self.model_path)
    return self

  def embed(self, word:str):
    # Predict the embedding of the word
    return self.model[word]

  def similar_words(self, word:str):
    return self.model.get_nearest_neighbors(word)



In [ ]:
med_train = f"{MED_DIR}/QUAERO_FrenchMed_traindev.ospl"
press_train = f"{PRESS_DIR}/QUAERO_FrenchPress_traindev.ospl"

Med_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_med.bin").fit(
    train_set=med_train)
Press_Emb = FastText_Embedding(model_path=f"{OUTPUT_DIR}/fasttext_press.bin").fit(
    train_set=press_train)


Read 0M words
Number of words:  9105
Number of labels: 0
Progress: 100.0% words/sec/thread:   92780 lr:  0.000000 avg.loss:  3.576015 ETA:   0h 0m 0s
Read 1M words
Number of words:  39655
Number of labels: 0
Progress: 100.0% words/sec/thread:   92945 lr:  0.000000 avg.loss:  2.304385 ETA:   0h 0m 0s


In [ ]:
print("Med vocabulary size:", len(Med_Emb.model.get_words()))
print("Press vocabulary size:", len(Press_Emb.model.get_words()))

Med vocabulary size: 9105
Press vocabulary size: 39655


In [ ]:
print(len(Med_Emb.embed("rideaux"))) # should return a 100-vector
print(len(Press_Emb.embed("rideaux"))) # should return a 100-vector

100
100


# Embeddings with WORD2VEC

In [ ]:
for p in [med_train, press_train]:
    print(("OK   " if os.path.exists(p) else "MISS "), p)
assert os.path.exists(med_train) and os.path.exists(press_train), "Input files not found - check DATA"

CORPORA = {"med": med_train, "press": press_train}

OUT_DIR = os.path.join(DATA_DIR, "word2vec_output")
EMB_DIR = os.path.join(OUT_DIR, "embeddings")
RESULTS_DIR = os.path.join(OUT_DIR, "results")
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Outputs will be saved to:", OUT_DIR)

OK    extracted_data/QUAERO_FrenchMed/QUAERO_FrenchMed_traindev.ospl
OK    extracted_data/QUAERO_FrenchPress/QUAERO_FrenchPress_traindev.ospl
Outputs will be saved to: extracted_data/word2vec_output


In [ ]:
# Show the first 3 raw lines of each corpus
for name, path in CORPORA.items():
    print(f"{name}: first 3 lines")
    with open(path, encoding="utf-8") as f:
        for _, line in zip(range(3), f):
            print(repr(line[:200]))
    print()

med: first 3 lines
'EMEA / H / C / 551 \n'
'PRIALT \n'
'Qu ’ est ce que Prialt ? \n'

press: first 3 lines
'Patricia Martin , que voici , que voilà ! oh , bonjour Nicolas Stoufflet . \n'
'France Inter , 7 heures . \n'
'le journal , Simon Tivolle . \n'



In [ ]:
# Corpus statistics: sentences, tokens, vocabulary size
counters, stats = {}, []
for name, path in CORPORA.items():
    cnt, n_sent = Counter(), 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            toks = line.split()
            if toks:
                n_sent += 1
                cnt.update(toks)
    counters[name] = cnt
    stats.append({"corpus": name, "sentences": n_sent, "tokens": sum(cnt.values()),
                  "vocabulary": len(cnt),
                  "words only seen once": sum(1 for c in cnt.values() if c == 1)})

corpus_stats = pd.DataFrame(stats).set_index("corpus")
corpus_stats.to_csv(os.path.join(RESULTS_DIR, "corpus_stats.csv"))
corpus_stats

,sentences,tokens,vocabulary,words only seen once
corpus,,,,
med,3021,51896,9104,5594
press,38548,1251561,39654,15402


In [ ]:
# Frequency of the 5 target words in each corpus
freq = pd.DataFrame({name: [counters[name][w] for w in WORDS] for name in CORPORA}, index=WORDS)
freq.index.name = "word"
freq.to_csv(os.path.join(RESULTS_DIR, "word_frequencies.csv"))
freq

,med,press
word,,
patient,33,11
traitement,251,53
maladie,65,114
solution,67,100
jaune,9,23


The Med corpus is small, and the Press corpus is approximately 24 times bigger. However, medical words appear more often in Med (traitement: 251 vs 53). Some words are very rare (jaune: 9 in Med), so their results may not be reliable.

# Training the 4 word2vec models

Key parameter sg (skipgram): with **sg=0** model becomes **CBOW**, that is, predicting the word from its context; with **sg=1** the model becomes **Skip-gram** - predict the context from the word

In [ ]:
PARAMS = dict(vector_size=100, min_count=1, window=5, epochs=5, workers=4, seed=42)
ALGOS = {"cbow": 0, "skipgram": 1}

models, train_log = {}, []
# first time it trains on (med, cbow), then (med, skipgram), then (press, cbow)
# and finally (press, skipgram)
for corpus, path in CORPORA.items():
    for algo, sg in ALGOS.items():
        key = f"w2v_{algo}_{corpus}"
        print(f"Training {key} ...")
        t0 = time.time()
        model = Word2Vec(sentences=LineSentence(path), sg=sg, **PARAMS)
        secs = round(time.time() - t0, 1)

        model.wv.save_word2vec_format(os.path.join(EMB_DIR, f"{key}.vec"), binary=False)

        models[key] = model
        train_log.append({"model": key, "corpus": corpus, "algo": algo, "sg": sg,
                          "vocab_size": len(model.wv), "corpus_tokens": model.corpus_total_words,
                          "train_seconds": secs})
        print(f"  done: vocab={len(model.wv)}, {secs}s")

train_df = pd.DataFrame(train_log).set_index("model")
# train_df.to_csv(os.path.join(RESULTS_DIR, "training_log.csv"))

train_df

Training w2v_cbow_med ...
  done: vocab=9104, 0.6s
Training w2v_skipgram_med ...
  done: vocab=9104, 0.9s
Training w2v_cbow_press ...
  done: vocab=39654, 8.0s
Training w2v_skipgram_press ...
  done: vocab=39654, 26.3s


,corpus,algo,sg,vocab_size,corpus_tokens,train_seconds
model,,,,,,
w2v_cbow_med,med,cbow,0,9104,51896,0.6
w2v_skipgram_med,med,skipgram,1,9104,51896,0.9
w2v_cbow_press,press,cbow,0,39654,1251561,8.0
w2v_skipgram_press,press,skipgram,1,39654,1251561,26.3


In [ ]:
# List the saved embedding files
for fn in sorted(os.listdir(EMB_DIR)):
    print(f"{fn:35s} {os.path.getsize(os.path.join(EMB_DIR, fn)) / 1e6:8.1f} MB")

w2v_cbow_med.vec                        11.7 MB
w2v_cbow_press.vec                      49.6 MB
w2v_skipgram_med.vec                    11.4 MB
w2v_skipgram_press.vec                  47.6 MB


# Evaluation: Semantic similarity

## Method: scipy distance

We chose to do it with scipy because fasttext was developed using the *fasttext* library instead of *gensim*

In [ ]:
models_w2v = models
models_fasttext = {}
models_fasttext["fasttext_Med_Emb"] = Med_Emb.model
models_fasttext["fasttext_Press_Emb"] = Press_Emb.model
print(models_w2v)
print(models_fasttext)

{'w2v_cbow_med': <gensim.models.word2vec.Word2Vec object at 0x77119102f3d0>, 'w2v_skipgram_med': <gensim.models.word2vec.Word2Vec object at 0x77119109bcd0>, 'w2v_cbow_press': <gensim.models.word2vec.Word2Vec object at 0x77118e5b7090>, 'w2v_skipgram_press': <gensim.models.word2vec.Word2Vec object at 0x77118e3dc410>}
{'fasttext_Med_Emb': <fasttext.FastText._FastText object at 0x7711c9807290>, 'fasttext_Press_Emb': <fasttext.FastText._FastText object at 0x7711c8b2fb90>}


In [ ]:
def closest_words_per_model(word, model, model_name):
    results = []

    if word in model.wv:
        vector = model.wv[word].reshape(1, -1)

        cosine_dist = distance.cdist(
            vector,
            model.wv.vectors,
            metric='cosine'
        )[0]

        sorted_indices = np.argsort(cosine_dist)[1:11]

        for rank, i in enumerate(sorted_indices, 1):
            closest_word = model.wv.index_to_key[i]

            results.append({
                'model_name': model_name,
                'target_word': word,
                'rank': rank,
                'closest_word': closest_word,
                'distance': cosine_dist[i]
            })

    else:
        results.append({
            'model_name': model_name,
            'target_word': word,
            'rank': np.nan,
            'closest_word': f"Word '{word}' not found",
            'distance': np.nan
        })

    return results


def closest_words_per_model_fasttext(
    word,
    model,
    model_name,
    all_words,
    all_vectors
):
    results = []

    if word in all_words:

        word_vector = model.get_word_vector(word).reshape(1, -1)

        cosine_dist = distance.cdist(
            word_vector,
            all_vectors,
            metric='cosine'
        )[0]

        sorted_indices = np.argsort(cosine_dist)[1:11]

        for rank, i in enumerate(sorted_indices, 1):

            results.append({
                'model_name': model_name,
                'target_word': word,
                'rank': rank,
                'closest_word': all_words[i],
                'distance': cosine_dist[i]
            })

    else:
        results.append({
            'model_name': model_name,
            'target_word': word,
            'rank': np.nan,
            'closest_word': f"Word '{word}' not found",
            'distance': np.nan
        })

    return results


all_results = []


# Word2Vec models

for model_name, model in models_w2v.items():

    for word in WORDS:

        all_results.extend(
            closest_words_per_model(
                word,
                model,
                model_name
            )
        )


# FastText models

for model_name, model in models_fasttext.items():

    all_words = model.get_words()

    all_vectors = np.array([
        model.get_word_vector(word)
        for word in all_words
    ])

    for word in WORDS:

        all_results.extend(
            closest_words_per_model_fasttext(
                word,
                model,
                model_name,
                all_words,
                all_vectors
            )
        )


results_df = pd.DataFrame(all_results)


### Corpus-based evaluation
Here we present the 10 closest words to the given words for each model, comparing the models based on the corpus it was trained on. On the left we see models trained on the medical corpus and on the right the ones trained on the press corpus.

In [ ]:
from IPython.display import display_html

models_med = ['w2v_cbow_med', 'w2v_skipgram_med', 'fasttext_Med_Emb']
models_press = ['w2v_cbow_press', 'w2v_skipgram_press', 'fasttext_Press_Emb']

# Prepare the data for pivoting
results_df['closest_word_distance'] = (
    results_df['closest_word']
    + ' ('
    + results_df['distance'].round(4).astype(str)
    + ')'
)

for word in WORDS:
    print(f"### Closest words for '{word}'\n")

    word_df = results_df[results_df['target_word'] == word]

    # Create pivot tables
    pivot_med = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_press = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    # add the med-trained models next to the press-trained ones for easy comparison
    med_columns = [m for m in models_med if m in pivot_med.columns]
    press_columns = [m for m in models_press if m in pivot_press.columns]

    pivot_med = pivot_med[med_columns].fillna('N/A')
    pivot_press = pivot_press[press_columns].fillna('N/A')

    # Add corpus labels above the tables
    html = f"""
    <div style="display: flex; gap: 30px; align-items: flex-start;">
        <div>
            <h4 style="text-align: center;">Medical Corpus</h4>
            {pivot_med.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Press Corpus</h4>
            {pivot_press.to_html()}
        </div>
    </div>
    """

    display_html(html, raw=True)
    print("\n")

### Closest words for 'patient'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,sont (0.0006),non (0.0029),patiente (0.0)
2,les (0.0006),réactions (0.003),patients (0.0)
3,le (0.0007),Tysabri (0.0034),appartient (0.0)
4,que (0.0007),arrêt (0.0036),entrainement (0.0)
5,et (0.0007),qui (0.0038),département (0.0)
6,par (0.0007),associé (0.0038),conscient (0.0)
7,est (0.0007),produit (0.0038),isolement (0.0)
8,en (0.0007),médecins (0.004),Traitement (0.0)
9,à (0.0007),recommandé (0.004),Département (0.0)




### Closest words for 'traitement'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,que (0.0004),médicaments (0.0257),avancement (0.0)
2,Le (0.0004),TYSABRI (0.0271),département (0.0)
3,avec (0.0004),par (0.0273),Département (0.0)
4,les (0.0004),association (0.0277),traitements (0.0)
5,TYSABRI (0.0004),autres (0.0305),Traitement (0.0)
6,des (0.0004),utilisation (0.0313),ajustement (0.0)
7,en (0.0004),Le (0.0317),principalement (0.0)
8,dans (0.0004),maladie (0.0323),Ajustements (0.0)
9,chez (0.0005),le (0.0333),chimiquement (0.0)




### Closest words for 'maladie'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,du (0.0005),infection (0.0057),préopératoires (0.0)
2,la (0.0005),cancer (0.0065),musculaires (0.0)
3,le (0.0005),association (0.0069),péricentriolaire (0.0)
4,de (0.0005),administration (0.0076),conduire (0.0)
5,. (0.0005),en (0.0078),territoires (0.0)
6,dans (0.0005),charge (0.0084),circulatoires (0.0)
7,des (0.0005),VIH (0.0087),postopératoires (0.0)
8,à (0.0005),adulte (0.0089),secondaires (0.0)
9,avec (0.0005),homme (0.0092),ventriculaires (0.0)




### Closest words for 'solution'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,perfusion (0.0008),jour (0.0179),solutions (0.0)
2,pour (0.001),flacon (0.0182),Dissolution (0.0)
3,ou (0.0011),fois (0.0186),evolution (0.0)
4,2 (0.0011),100 (0.0205),Solution (0.0)
5,flacon (0.0011),contient (0.0213),exception (0.0)
6,/ (0.0011),150 (0.0235),contraception (0.0)
7,( (0.0011),mg (0.0239),exécution (0.0)
8,) (0.0011),300 (0.0243),congélation (0.0)
9,dose (0.0011),kg (0.0272),Evolution (0.0)




### Closest words for 'jaune'



model_name,w2v_cbow_med,w2v_skipgram_med,fasttext_Med_Emb
rank,,,
1,) (0.0025),14 (0.0015),identique (0.0)
2,. (0.0026),23 (0.0016),orientables (0.0)
3,: (0.0026),64 (0.0022),neuropathique (0.0)
4,( (0.0026),eau (0.0023),bicompartimental (0.0)
5,la (0.0027),Fluoxétine (0.0024),diagnostique (0.0)
6,dans (0.0027),corporel (0.0025),typiquement (0.0)
7,", (0.0027)",nystagmus (0.0026),intention (0.0)
8,• (0.0027),421 (0.0026),Franceville (0.0)
9,3 (0.0027),13 (0.0027),historique (0.0)


Interpretation: we can see that the model has created very similar embeddings for words in the medical domain (for the medical corpus) and words in the press domain. For example, for 'patient', we can see that the closest word in the medical domain, the skipgram and fasttext give words like 'symptômes', 'recommandé', 'Traitement' and so on. Meanwhile, the same models trained on a press corpus give words that could be more likely to see in news articles, like 'lecteur', or 'idéologie'. There's a notable difference with CBOW, which returns shorts or stopwords as closest words for any given word. But, in general, the closest words returned per corpus are words that are expected in each specific domain.

### Model-based evaluation
Same results but aggregated by model type.

In [ ]:
models_cbow = ['w2v_cbow_med', 'w2v_cbow_press']
models_skipgram = ['w2v_skipgram_med', 'w2v_skipgram_press']
models_fasttext = ['fasttext_Med_Emb', 'fasttext_Press_Emb']

# Prepare the data for pivoting
results_df['closest_word_distance'] = (
    results_df['closest_word']
    + ' ('
    + results_df['distance'].round(4).astype(str)
    + ')'
)

for word in WORDS:
    print(f"### Closest words for '{word}'\n")

    word_df = results_df[results_df['target_word'] == word]

    # Create pivot tables
    pivot_cbow = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_skipgram = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    pivot_fasttext = word_df.pivot_table(
        index='rank',
        columns='model_name',
        values='closest_word_distance',
        aggfunc='first'
    )

    # add the med-trained models next to the press-trained ones for easy comparison
    cbow_columns = [m for m in models_cbow if m in pivot_cbow.columns]
    skipgram_columns = [m for m in models_skipgram if m in pivot_skipgram.columns]
    fasttext_columns = [m for m in models_fasttext if m in pivot_fasttext.columns]

    pivot_cbow = pivot_cbow[cbow_columns].fillna('N/A')
    pivot_skipgram = pivot_skipgram[skipgram_columns].fillna('N/A')
    pivot_fasttext = pivot_fasttext[fasttext_columns].fillna('N/A')

    # Add corpus labels above the tables
    html = f"""
    <div style="display: flex; gap: 30px; align-items: flex-start;">
        <div>
            <h4 style="text-align: center;">Models using Continuous Bag of Words (CBOW)</h4>
            {pivot_cbow.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Models using skipgram</h4>
            {pivot_skipgram.to_html()}
        </div>
        <div>
            <h4 style="text-align: center;">Press Corpus</h4>
            {pivot_fasttext.to_html()}
        </div>
    </div>
    """

    display_html(html, raw=True)
    print("\n")

### Closest words for 'patient'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,sont (0.0006),consensus (0.0919)
2,les (0.0006),geste (0.1048)
3,le (0.0007),effort (0.1182)
4,que (0.0007),désir (0.1186)
5,et (0.0007),déluge (0.1223)
6,par (0.0007),succès (0.1264)
7,est (0.0007),déballage (0.1294)
8,en (0.0007),parcours (0.1311)
9,à (0.0007),avertissement (0.1322)




### Closest words for 'traitement'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,que (0.0004),laboratoire (0.0682)
2,Le (0.0004),tabac (0.0734)
3,avec (0.0004),contrat (0.0778)
4,les (0.0004),matériel (0.0785)
5,TYSABRI (0.0004),prophète (0.0802)
6,des (0.0004),lycée (0.0815)
7,en (0.0004),mariage (0.0823)
8,dans (0.0004),linge (0.0831)
9,chez (0.0005),roman (0.0833)




### Closest words for 'maladie'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,du (0.0005),exacte (0.0852)
2,la (0.0005),polémique (0.0866)
3,le (0.0005),population (0.0867)
4,de (0.0005),puissance (0.0877)
5,. (0.0005),procédure (0.0891)
6,dans (0.0005),capacité (0.0942)
7,des (0.0005),menace (0.0966)
8,à (0.0005),méthode (0.1006)
9,avec (0.0005),négociation (0.1007)




### Closest words for 'solution'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,perfusion (0.0008),puissance (0.081)
2,pour (0.001),meilleure (0.0868)
3,ou (0.0011),portée (0.0933)
4,2 (0.0011),satisfaction (0.0973)
5,flacon (0.0011),seule (0.0977)
6,/ (0.0011),forme (0.0982)
7,( (0.0011),formule (0.0992)
8,) (0.0011),couverture (0.0996)
9,dose (0.0011),règle (0.0998)




### Closest words for 'jaune'



model_name,w2v_cbow_med,w2v_cbow_press
rank,,
1,) (0.0025),cavalier (0.0456)
2,. (0.0026),maillot (0.0501)
3,: (0.0026),leader (0.0505)
4,( (0.0026),Nouvel (0.0552)
5,la (0.0027),dark (0.0552)
6,dans (0.0027),indomptable (0.0558)
7,", (0.0027)",dix-septième (0.0576)
8,• (0.0027),parisien (0.0586)
9,3 (0.0027),constructeur (0.059)


Interpretation: here it is easier to compare the behavior of each model. For example, we can detect that fasttext tends to associate words that have similar ending to the given word, since it represents words using information learned from subwords. And cbow returns words or characters that tend to happen nearby, like skipgram, although they learn contextual information differently.

We notice that fasttext tends to produce smaller distances for each word (distance of 0 in some cases). Models trained on CBOW provide also close distances but less than fasttext. And finally skipgram models' embeddings show larger distances to their closest neighbors.

Another common pattern can be observed between corpora. The embeddings of all models trained on the medical domain report closer distances than the same model trained on the press domain. This can be due to the bigger size of the press corpus (it contains over 4 times the amount of words present in the medical corpus, and over 3 times the amout of unique words). This may result in less concentrated embedding space and therefore larger distances between target words and nearest neighbors.  

## ML Classification NER

In [ ]:
TRAIN_DATA_PATH = os.path.join(DATA_DIR, "QUAERO_FrenchMed/EMEA/EMEAtrain_layer1_ID.conll")
VALID_DATA_PATH = os.path.join(DATA_DIR, "QUAERO_FrenchMed/EMEA/EMEAdev_layer1_ID.conll")
TEST_DATA_PATH = os.path.join(DATA_DIR, "QUAERO_FrenchMed/EMEA/EMEAtest_layer1_ID.conll")

In [ ]:
emb_model_path = [f"{OUTPUT_DIR}/fasttext_med.bin",
                  f"{EMB_DIR}/w2v_cbow_med.vec",
                  f"{EMB_DIR}/w2v_skipgram_med.vec "]

for model in ["cnn", "lstm"]:
    for idx, model_path in enumerate(emb_model_path):
        resultat = subprocess.run(
            [
                "python", 
                "scripts/cnn_classification.py", 
                "--model", model, 
                "--emb-model", idx, 
                "--emb-model-path", model_path, 
                "--train", TRAIN_DATA_PATH, 
                "--valid", VALID_DATA_PATH, 
                "--test", TEST_DATA_PATH, 
                "--epochs", "10"
            ], 
            capture_output=True, 
            text=True
        )

        print(resultat.stdout)

: 